In [1]:
# Import packages
import os
import time
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)

# Stops ID (v0.1)

A notebook to lay out the logic of the Stops Identification process, in preparation for their conversion to voyages. The algorithms are designed to combine the raw AIS dataset with a suitable dataset of Port locations, ultimately resulting in a separate table of voyages associated with the AIS dataset. The geospatial processing is conducted using 'S3' due to it's computationally effective approach to geospatial analysis. 

<u><h5>Input</h5></u>

Uses output of the 'Setup' script. 

<u><h5>1. ...</h5></u>

...

<u><h5>Output</h5></u>

A standalone dataset, separate to the EFG process, but which can be used to inform it. 

A future version of this notebook may seek to incorporate the following:

<ul>
    <li>.</li>
</ul>

## 1. Existing Method from Wendela's PhD

The method documented in Schim van der Loeff identifies port stops via the following methodology. First, a couple of pre-processing steps are executed:

<ul>
    <li>Each of the 1.8bn hourly AIS points (across 3 years) is assigned to its nearest port using an efficient 'spatial nearest neighbour algorithm' applied to a ports dataset with around 13,000 ports in it. The distance to this port is also recorded.</li>
    <li>All contiguous messages that indicate a vessel is at a standstill (with a SOG < 1 knot) are grouped together and treated as a 'cluster'. Each cluster is assigned to the closest port identified in the previous step.</li>
    <li>An additional condition is introduced to identify clusters to account for cases where AIS coverage is too poor to enable a SOG-based algorithm to work. Specifically, a cluster is identified when a vessel remains within 15 nm of a port for more than 6 hours or within 5 nm for more than 3 hours, but only if the clusters messages were interpolated. These clusters then undergo the same filtering process as the clusters identified above.</li>
</ul>

<u><h5>Conversion of Clusters to 'Stops'</h5></u>

Then clusters undergo a series of merging and filtering steps to transform them into 'stops'.

<ol>
    <li>Firstly, clusters more than 25nm from a port are removed (with some exceptions), set fairly large in order to accommodate poor AIS coverage and the fact that ports may be large complexes yet only represented by a single centroid.</li>
    <li>Consecutive clusters located near each other and matched to the same port are merged into one cluster.</li>
    <li>If the distance between two consecutive stops is less than 30 nm but the identified ports are different (a situation that can occur due to congestion on the way to a port or when a vessel waits close to another port whilst waiting for entry), one of the clusters is removed based on which of the two ports appears more frequently in the vessel's entire trajectory.</li>
</ol>

So, to identify a cluster of AIS messages as a stop, the following criteria need to be true: i. the distance to its nearest port is sufficiently small; ii. the time spent at port is sufficiently large; and iii. the time intervals between the cluster and both its neighbouring clusters are sufficiently large.

<u><h5>Area-specific Adjustment</h5></u>

In addition, area-specific adjustments to the algorithm are also implemented in order to avoid spurious or excessive vessel stop identifications  Such areas include the Panama, Suez and Kiel canals and the Straits of Gibralter, Singapore and the Bosphorus. Many vessels reduce their speed when approaching these canals and straits and may spend time idling near port before proceeding, especially during busy periods. A minimum of 24 hours is required for a port stop in the Suez and Panama canals, and 12 hours in the Kiel area, however such methods can't stop excessive identifications if a canal or strait is blocked for an extended period of time. For the Bosphorus, 24 hours are required and a more stringent 10 nm buffer is required rather than 25nm. 

<u><h5>Application</h5></u>

Identified vessel stops are then used to estimate voyages and voyage parameters, including duration and distance sailed, where the stop is considered part of its preceding voyage rather than subsequent one. 'Normalised' voyage- and stop-specific draughts are also evaluated using a combination of observed and infilled instantaneous draught values, and median AIS-derived draught during voyage and at stop.

## 2. Method for the UN Global Platform

Wendela's method seeks to assign the id and distance to the nearest port associated with every hourly AIS position. Given there are around 13,000 ports in the database, this is likely to represent a heavily computationally intensive approach. 

Instead, LMS.4 already contains pre-existing datasets representing 1 nm and 5 nm Coastal and Port buffers in H3. These can be read in with the following code:

```python
coast_1 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_coast_1nm.csv"))
coast_5 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_coast_5nm.csv"))
port_1 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_wpi_port_polys_1NM.csv"))
port_5 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_wpi_port_polys_5NM.csv"))
#port_25 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_wpi_port_polys_25NM.csv"))
```

Therefore, for an initial version, let's use the 'port_5' polygon to analyse the AIS dataset for clusters. However, to align with Wendela's method, it would be useful to derive a 'port_25' dataset.

<u><h5>Establishing the 'Cluster' Dataset</h5></u>

Iterating for every vessel, the AIS data falling within the port buffer geometry is pulled. A 'clusters' dataset is then established to track contiguous AIS messages recording a SOG of < 1 kn (in alignment with Wendela's method). It's my understanding that there hasn't been any infilling applied to the AIS data hosted on the UN Global Platform, therefore the 'additional condition' (based on distance to port alone) isn't included here. 

The closest port and distance to that port is then added.

<u><h5>Conversion of Clusters to 'Stops'</h5></u>

Then clusters undergo a series of merging and filtering steps to transform them into 'stops'.

<ol>
    <li>Consecutive clusters located near each other and matched to the same port are merged into one cluster.</li>
    <li>If the distance between two consecutive stops is less than 30 nm but the identified ports are different (a situation that can occur due to congestion on the way to a port or when a vessel waits close to another port whilst waiting for entry), one of the clusters is removed based on which of the two ports appears more frequently in the vessel's entire trajectory.</li>
</ol>

<u><h5>Application</h5></u>

Identified vessel stops are then used to estimate voyages and voyage parameters, including duration and distance sailed, where the stop is considered part of its preceding voyage rather than subsequent one.

## Input

In [1]:
port_5 = spark.createDataFrame(ef.find_folder_csv("polygons","h3_wpi_port_polys_5NM.csv"))
port_5.iloc[:4]

## 1. 

## Output